# 教材简介整理助手

Axiom 配套实验。请先读懂对应教材，再按三个阶段练习。计算在你自己的电脑运行。

## 准备环境

建议 Python 3.11 或 3.12，在单独虚拟环境中运行下面的安装命令，然后用 `jupyter notebook` 打开本文件。

```bash
python -m pip install notebook 
```

先运行“跟着做”。带 TODO 的函数初始返回 None，检查单元会提示待完成。补全后重新运行函数定义与检查单元。独立练习应另写实现，再核对结果。参考解答在文件末尾；网页的阶段勾选是个人学习记录，不代表系统已执行你的代码。

按 Hands-On Large Language Models 的 Prompt Engineering 章。联网可能产生服务商费用。Notebook 默认不联网、不保存密钥；离线结果不能冒充真实模型实验。

In [ ]:
import json,os,getpass,urllib.request
# 虚构教学材料；默认只处理保存的示例响应，不会连接或计费。
INTRO='书名：线性代数练习。主题：矩阵、线性方程组。需要基础：基本代数运算。'
SAMPLE_REPLY='{"title":"线性代数练习","topics":["矩阵","线性方程组"],"prerequisites":["基本代数运算"]}'
UNKNOWN_REPLY='{"title":"分析札记","topics":["导数"],"prerequisites":null}'
def call_model(prompt):
    # 只有你明确调用本函数才会联网。不要把管理员默认密钥复制到电脑。
    base=os.environ.get('AXIOM_AI_BASE_URL','').rstrip('/')
    model=os.environ.get('AXIOM_AI_MODEL','')
    if not base or not model:raise ValueError('先设置自己的 AXIOM_AI_BASE_URL 与 AXIOM_AI_MODEL')
    if not base.startswith('https://'):raise ValueError('真实调用请使用服务商的 HTTPS 兼容 API 地址')
    key=os.environ.get('AXIOM_AI_API_KEY') or getpass.getpass('你的个人 API 密钥（不回显）：')
    payload={'model':model,'messages':[{'role':'user','content':prompt}]}
    request=urllib.request.Request(base+'/chat/completions',data=json.dumps(payload).encode(),headers={'Content-Type':'application/json','Authorization':'Bearer '+key},method='POST')
    with urllib.request.urlopen(request,timeout=60) as response:body=json.load(response)
    text=body.get('choices',[{}])[0].get('message',{}).get('content')
    if not isinstance(text,str):raise ValueError('模型未返回文本回答，请检查模型与接口格式')
    return text
def check_parser(parser):
    card=parser(SAMPLE_REPLY)
    assert card==json.loads(SAMPLE_REPLY)
    assert parser(UNKNOWN_REPLY)['prerequisites'] is None
    for bad in ['not json','{"title":1,"topics":[],"prerequisites":null}','{"title":"x","topics":"matrix","prerequisites":null}']:
        try:parser(bad)
        except (ValueError,TypeError):pass
        else:raise AssertionError('应拒绝错误格式')
    print('格式检查通过；内容是否来自原文仍需人工逐项核对。')

## 1. 跟着做

逐步运行示例，解释每个输入、运算和输出。再修改一个输入观察变化。

In [ ]:
def demo_parser(raw):
    card=json.loads(raw)
    if not isinstance(card,dict) or set(card)!={'title','topics','prerequisites'}:raise ValueError('字段不完整或存在额外字段')
    if card['title'] is not None and not isinstance(card['title'],str):raise ValueError('title 必须为文本或 null')
    for name in ['topics','prerequisites']:
        value=card[name]
        if value is not None and (not isinstance(value,list) or any(not isinstance(item,str) for item in value)):raise ValueError(name+' 必须为文本列表或 null')
    return card
PROMPT='只根据简介抽取 title、topics、prerequisites。只输出 JSON。title 为文本或 null，另两项为文本列表或 null；未提及的内容填 null，不猜测。简介：'+INTRO
print('当前模式：离线示例响应，尚未调用真实模型。')
print(demo_parser(SAMPLE_REPLY))
# 真实调用二选一：
# 1. 在平台教材阅读器用管理员默认 API 提问，将原始回答粘贴进变量再解析。
# 2. 配好你自己的 API 后，主动执行：raw_reply=call_model(PROMPT)

## 2. 自己补全

在下面函数中填写 TODO。可回看示例，但请说明你填写的每一行做了什么。

In [ ]:
def student_parser(raw):
    card=json.loads(raw)
    # TODO 1: 检查字段恰好是 title、topics、prerequisites
    # TODO 2: title 是文本或 None，另两项是文本列表或 None
    # TODO 3: 格式错误时抛出 ValueError，通过时返回 card
    return None

In [ ]:
if student_parser(SAMPLE_REPLY) is None:print('待完成：补全格式检查。')
else:check_parser(student_parser)

## 3. 独立完成

换成简介“书名：分析札记。主题：导数。”，写出提示词并检查结果。原文没有给出先修条件，必须保留 null。先用示例响应检查程序，再完成一次真实模型调用并记录原文、提示与原始回答；无 API 时只能标记程序部分完成。

In [ ]:
def independent_card(raw):
    # TODO: 使用自己实现的解析器处理回答，并核对本次已知的内容
    return None

In [ ]:
card=independent_card(UNKNOWN_REPLY)
if card is None:print('待完成：独立核对新简介；真实 API 实验另行完成。')
else:
    assert card['title']=='分析札记' and card['topics']==['导数'] and card['prerequisites'] is None
    print('离线变式通过；这不表示已经验证真实模型。')

## 提示与复盘

先检查输入形状、数据类型和边界情况。网页“查看提示”提供针对本实验的提示。记录你遇到的一处错误、修复方法和结果。通过数值检查后，还需要自己解释原理。

## 参考解答

先独立尝试再展开下方代码。示例是可核对的一种实现，不是唯一写法。

In [ ]:
check_parser(demo_parser)
assert demo_parser(UNKNOWN_REPLY)['prerequisites'] is None
print('参考提示：只抽取原文明确给出的信息，未提及的 prerequisites 必须为 null。')